# Evaluation Notebook
## AI Competitive Intelligence Copilot — Luxury Fashion

Evaluates three metrics from the spec:
1. **Relevance Precision@10** — top 10 retrieved items: % useful
2. **Event Classification F1** — model vs human-labeled event types
3. **Trend Precision@5** — top 5 trends: % judged valid

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.db import get_connection, get_latest_trends
from src.evaluation import precision_at_k, event_f1, trend_precision_at_k, EVENT_TYPES

## Load data from DB

In [ ]:
conn = get_connection()

# Load top 10 items by impact score
df_top10 = pd.read_sql_query("""
    SELECT i.item_id, i.competitor, i.source_name, i.title, i.source_url,
           e.event_type, e.impact_score, e.relevance_score, e.summary
    FROM items i
    JOIN events e ON i.item_id = e.item_id
    ORDER BY e.impact_score DESC
    LIMIT 10
""", conn)

# Load all events for F1 evaluation
df_all_events = pd.read_sql_query("""
    SELECT i.item_id, i.title, i.competitor, i.source_url,
           e.event_type, e.confidence_score, e.evidence_snippet, e.summary
    FROM events e
    JOIN items i ON e.item_id = i.item_id
    LIMIT 30
""", conn)

conn.close()
print(f"Loaded {len(df_top10)} top items and {len(df_all_events)} events for evaluation")

## Metric 1: Relevance Precision@10

Manually label each of the top 10 items as relevant (1) or not (0).

**Relevant** = the article genuinely discusses a competitive signal for Chanel, Dior, or Gucci.

Run the cell below, review the items, then fill in `human_labels`.

In [ ]:
# Display top 10 for manual review
pd.set_option('display.max_colwidth', 80)
df_top10[['competitor', 'event_type', 'impact_score', 'title', 'source_name']].style.background_gradient(subset=['impact_score'], cmap='RdYlGn')

In [ ]:
# ── FILL IN: 1 = relevant, 0 = not relevant (one per row above) ──
human_labels_p10 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]  # edit this after reviewing

precision_at_10 = precision_at_k(human_labels_p10, k=10)
print(f"Relevance Precision@10: {precision_at_10:.0%}  ({sum(human_labels_p10)}/{len(human_labels_p10)} relevant)")

## Metric 2: Event Classification F1

Review a sample of events and provide the correct event type label.
Then compute F1 against the model's predictions.

In [ ]:
# Display events for labeling
for i, row in df_all_events.head(20).iterrows():
    print(f"[{i:02d}] {row['competitor']:8s} | Model: {row['event_type']:<35s} | {row['title'][:70]}")

In [ ]:
# Model predictions (from DB)
y_pred = df_all_events.head(20)['event_type'].tolist()

# ── FILL IN: your human labels for the 20 events above ──
y_true = y_pred.copy()  # replace with your annotations after reviewing

scores = event_f1(y_true, y_pred)
print(f"F1 Score (macro):    {scores['macro']:.3f}")
print(f"F1 Score (weighted): {scores['weighted']:.3f}")
print()
print(scores['report'])

f1_macro = scores['macro']

## Metric 3: Trend Precision@5

Review the top 5 detected trends and judge whether each is a valid strategic signal.

In [ ]:
trends = get_latest_trends(limit=5)
df_trends = pd.DataFrame(trends)

if df_trends.empty:
    print("No trends found. Run the pipeline first.")
else:
    display_cols = ['competitor', 'event_type', 'trend_score', 'count_7d', 'unique_sources', 'avg_impact', 'is_critical']
    df_trends[display_cols].style.background_gradient(subset=['trend_score'], cmap='Blues')

In [ ]:
# ── FILL IN: 1 = valid trend, 0 = false positive (one per row above) ──
human_trend_labels = [1, 1, 1, 1, 1]  # edit after reviewing

trend_precision_at_5 = trend_precision_at_k(human_trend_labels, k=5)
print(f"Trend Precision@5: {trend_precision_at_5:.0%}  ({sum(human_trend_labels[:5])}/{min(len(human_trend_labels), 5)} valid)")

## Summary Dashboard

In [ ]:
import matplotlib.pyplot as plt

metrics = {
    'Relevance\nPrecision@10': precision_at_10,
    'Event F1\n(macro)': f1_macro,
    'Trend\nPrecision@5': trend_precision_at_5,
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(metrics.keys(), metrics.values(), color=['#2c7bb6', '#d7191c', '#1a9641'], width=0.5)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Evaluation Metrics — AI Competitive Intelligence Copilot')
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f'{val:.0%}', ha='center', fontweight='bold')
ax.axhline(0.7, color='gray', linestyle='--', linewidth=0.8, label='Target (70%)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to data/evaluation_metrics.png")